# Module 5.0: From Corpus to Token Stream

In **Module 4.4** you assembled a working GPT. It has one appetite: a tensor of
integers, shape `(batch, block_size)`.

What you actually have is a **text file**.

Everything between those two facts is this notebook — and it is the part of LLM
training that papers compress into a paragraph and courses skip entirely. That's a
strange omission, because in a real pretraining project **the data pipeline is where
most of the engineering effort goes**, and it's where most of the quality comes from.
Two labs with the same architecture and the same compute budget will produce very
different models if one of them has better data.

### What you'll build

A complete, runnable pipeline:

```mermaid
flowchart LR
    A["raw text"] --> B["quality<br/>filter"]
    B --> C["dedup"]
    C --> D["train BPE<br/>(Module 2.1)"]
    D --> E["encode to<br/>uint16 stream"]
    E --> F["pack + split"]
    F --> G["get_batch<br/>-> (B, T) tensor"]
    G --> H["your GPT"]
```

### Why it belongs *here*

- It **finally uses the BPE tokenizer you built in Module 2.1**. Until now that was a
  toy on the word "lowest"; here you train it on a real corpus and encode a million
  characters with it.
- **Module 5.6 (evaluation)** is a lie without it. You cannot trust a held-out score
  until you know your test set isn't secretly in your training set — the failure mode
  called *contamination*, which we'll detect at the end of this notebook.
- **Module 5.7 (scaling laws)** is a law about *tokens*. This is where a "token"
  stops being an abstraction and becomes a number in an array.

**Prerequisites:** Module 2.1 (tokenization & BPE), Module 4.4 (you have a model to
feed).

## 1. Where does a trillion tokens actually come from?

Nobody hand-writes a pretraining corpus. The modern lineage looks like this:

| Layer | What it is | Scale |
|---|---|---|
| **Common Crawl** | a non-profit that has been scraping the public web since 2008, released as raw monthly dumps | petabytes of HTML |
| **Cleaned derivatives** | the crawl, filtered and de-duplicated by someone else so you don't have to: C4, RefinedWeb, **FineWeb** | 10–100 T tokens |
| **Curated additions** | code (GitHub), books, papers (arXiv), encyclopedic text (Wikipedia), Q&A (StackExchange) — collections like **The Pile** bundle these | 100 B – 1 T tokens |
| **Your mixture** | the recipe: how much of each, sampled in what proportion | the actual decision |

That last row is the interesting one. You don't train on "the internet" — you train on
a **mixture**, and the proportions are a design choice with measurable consequences.
A rough, representative recipe:

| Source | Share | Why it's in there |
|---|---:|---|
| Filtered web | ~65% | scale and breadth; nothing else is this big |
| Code | ~15% | improves *reasoning*, not just coding — structured, verifiable text |
| Books | ~10% | long-range coherence that web pages never have |
| Wikipedia / reference | ~5% | dense, high-quality facts |
| Papers, Q&A, other | ~5% | technical register and domain depth |

Two things worth internalising:

1. **Upsampling is normal.** Wikipedia is tiny compared to the web, so it gets
   *repeated* several times per epoch to hit its target share. Quality buys repetition.
2. **Code helps non-code tasks.** This surprised people, and it's now a standard
   ingredient in every frontier mixture.

For this notebook our "internet" is TinyShakespeare — small enough to run in seconds,
big enough that every step below is real.

In [ ]:
import os
import urllib.request
from collections import Counter

data_path = "../tinyshakespeare.txt"
data_url = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
            "master/data/tinyshakespeare/input.txt")

# The same load-with-fallback pattern the capstone (Module 5.3) uses, so this
# notebook never dead-ends on a missing network.
FALLBACK_TEXT = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?
""" * 400

if os.path.exists(data_path):
    text = open(data_path, encoding="utf-8").read()
    print(f"Loaded local '{data_path}'.")
else:
    try:
        text = urllib.request.urlopen(data_url, timeout=20).read().decode("utf-8")
        print("Downloaded TinyShakespeare.")
    except Exception as e:
        text = FALLBACK_TEXT
        print(f"Network unavailable ({type(e).__name__}); using the embedded fallback.")

print(f"\ncorpus: {len(text):,} characters, {len(text.split()):,} words, "
      f"{len(set(text.split())):,} unique words")
print(f"distinct characters: {len(set(text))}")
print("\n--- first 200 characters ---")
print(text[:200])

## 2. Quality filtering

Raw web text is *mostly garbage*: navigation bars, cookie banners, SEO spam, lorem
ipsum, broken encodings. Feeding it to a model doesn't just waste compute — it teaches
the model to produce that same garbage.

Real pipelines run dozens of filters. They are almost all boring heuristics, and the
boringness is the point: they must run over billions of documents, so nothing
expensive is allowed.

### The classic heuristics

| Filter | Rejects |
|---|---|
| Length | documents too short to contain an idea |
| Mean word length | gibberish, base64 blobs, minified JS |
| Symbol-to-word ratio | tables of numbers, ASCII art, code dumps in a prose corpus |
| Stop-word presence | text that isn't actually natural language |
| Line-level dedup | nav bars and footers repeated on every page |
| Perplexity (model-based) | text a small reference model finds bizarre |

Let's implement a few and watch them fire on documents designed to trip them.

In [ ]:
def quality_report(doc):
    """Return (passed, reason). Cheap heuristics only -- these must run on billions of docs."""
    words = doc.split()

    if len(words) < 10:
        return False, "too short (<10 words)"

    mean_word_len = sum(len(w) for w in words) / len(words)
    if not 3 <= mean_word_len <= 10:
        return False, f"mean word length {mean_word_len:.1f} outside [3, 10]"

    symbols = sum(1 for ch in doc if not ch.isalnum() and not ch.isspace())
    if symbols / max(len(words), 1) > 2.0:
        return False, f"symbol/word ratio {symbols / len(words):.1f} too high"

    # Real natural language nearly always contains common function words.
    STOP = {"the", "be", "to", "of", "and", "a", "in", "that", "have", "it", "is", "we"}
    if not any(w.lower().strip(".,!?;:") in STOP for w in words):
        return False, "no stop words -- probably not prose"

    return True, "ok"


candidates = {
    "good prose":   "It is the east and the sky is bright with a light that we have not seen "
                    "before in all of our long years upon this shore.",
    "too short":    "Click here now.",
    "base64 blob":  "aGVsbG8gd29ybGQgdGhpcyBpcyBub3QgcmVhbCB0ZXh0IGF0IGFsbA== " * 4,
    "symbol soup":  "| --- | --- | >>> ### $$$ {{}} [[]] ||| @@@ %%% ^^^ &&& *** ((( ))) " * 3,
    "keyword spam": "cheap flights cheap hotels cheap cars discount discount discount "
                    "booking booking booking travel travel travel deals deals deals",
}

print(f"{'document':<15} {'verdict':<8} reason")
print("-" * 62)
for name, doc in candidates.items():
    ok, reason = quality_report(doc)
    print(f"{name:<15} {'KEEP' if ok else 'DROP':<8} {reason}")

print("""
Read those reasons carefully -- two of them are luck:

  * 'base64 blob' was dropped for being TOO SHORT, not for being base64. It has no
    spaces, so .split() sees 4 "words". The right filter fired for the wrong reason.
  * 'keyword spam' was dropped for having no stop words -- but add one "the" and it
    sails through, because it is otherwise perfectly well-formed.

That is the honest character of heuristic filtering: cheap, useful, and full of holes.
It is a coarse first pass. Deduplication (next) is what actually kills spam at scale,
because spam is, by its nature, repeated.""")

## 3. Deduplication — the highest-value step in the whole pipeline

If you only do one thing to your corpus, do this one.

The web is enormously redundant: the same article syndicated across 50 sites, the same
licence text in 100,000 repositories, the same product description with one word
changed. Duplicates hurt in three separate ways:

1. **Wasted compute.** A duplicate seen 10 times consumes 10× the FLOPs to teach the
   model one thing.
2. **Memorization.** Text repeated often enough stops being *learned* and starts being
   *recited* — which is how models leak training data verbatim.
3. **Broken evaluation.** If a duplicate straddles your train/test split, your held-out
   score is measuring memory, not generalization. (§8.)

There are two flavours, and you need both.

### Exact dedup: hash it

Trivial and cheap — hash every document, keep first occurrence.

### Near-dup: MinHash + Jaccard

Exact hashing misses "the same document with one word changed". The standard tool:

1. Cut each document into **shingles** (overlapping n-grams of words).
2. Two documents are near-duplicates if their shingle *sets* overlap a lot — measured
   by **Jaccard similarity**: $J(A,B) = \dfrac{|A \cap B|}{|A \cup B|}$.
3. Comparing every pair is $O(n^2)$ and impossible at scale, so **MinHash** compresses
   each set into a short signature whose collision probability *equals* the Jaccard
   similarity. Then you only compare signatures.

We'll build all three below — the exact version, the honest Jaccard, and a MinHash
that estimates it.

In [ ]:
import hashlib
import random

docs = [
    "the quick brown fox jumps over the lazy dog in the field",
    "the quick brown fox jumps over the lazy dog in the field",     # EXACT duplicate
    "the quick brown fox leaps over the lazy dog in the field",     # NEAR duplicate (1 word)
    "to be or not to be that is the question we must all answer",   # unrelated
    "the quick brown fox jumps over the lazy cat in the meadow",    # near-ish (2 words)
]

# --- Exact dedup: hash and keep first occurrence -----------------------------
seen, exact_kept = set(), []
for i, d in enumerate(docs):
    h = hashlib.sha256(d.encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        exact_kept.append(i)
print("Exact dedup keeps documents:", exact_kept, f"(dropped {len(docs) - len(exact_kept)})")

# --- Shingles + true Jaccard -------------------------------------------------
def shingles(doc, k=3):
    """Overlapping k-word windows. k=3 is a common choice for prose."""
    w = doc.split()
    return {" ".join(w[i:i + k]) for i in range(len(w) - k + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

print("\nTrue Jaccard similarity vs document 0:")
s0 = shingles(docs[0])
for i, d in enumerate(docs):
    print(f"  doc {i}: {jaccard(s0, shingles(d)):.3f}   {d[:48]}...")

In [ ]:
# --- MinHash: estimate that same Jaccard from a short signature --------------
# The trick: hash every shingle with N different hash functions and keep the MINIMUM
# for each. The probability that two sets share a minimum equals their Jaccard
# similarity -- so the fraction of matching signature slots ESTIMATES Jaccard,
# using N numbers instead of the whole set.

N_HASHES = 128
rng = random.Random(0)
# Each "hash function" is just a random salt mixed into a real hash.
SALTS = [rng.getrandbits(32) for _ in range(N_HASHES)]

def minhash(doc, k=3):
    sh = shingles(doc, k)
    sig = []
    for salt in SALTS:
        sig.append(min(hash((s, salt)) for s in sh) if sh else 0)
    return sig

def estimated_jaccard(sig_a, sig_b):
    return sum(x == y for x, y in zip(sig_a, sig_b)) / len(sig_a)

sig0 = minhash(docs[0])
print(f"{'doc':<5}{'true J':>9}{'MinHash est':>14}   verdict (threshold 0.5)")
print("-" * 58)
for i, d in enumerate(docs):
    true_j = jaccard(s0, shingles(d))
    est_j = estimated_jaccard(sig0, minhash(d))
    verdict = "NEAR-DUP of doc 0" if est_j > 0.5 and i != 0 else ""
    print(f"{i:<5}{true_j:>9.3f}{est_j:>14.3f}   {verdict}")

print(f"\nThe estimate tracks the truth using {N_HASHES} integers per document instead")
print("of the full shingle set -- that is what makes dedup possible at web scale.")

## 4. Train *your* BPE on the corpus

In **Module 2.1** you built Byte-Pair Encoding from scratch and ran it on a toy corpus
of `low / lower / lowest`. It has been sitting unused ever since. Time to cash it in.

The algorithm is identical — the only change is bookkeeping. In 2.1 we kept a `Counter`
of *tokenized* words; here we keep a **dict from the original word to its current
tokenization**, so that when training finishes we already have the encoding table for
every word in the corpus. Same merges, free lookup table.

Recall the loop:

1. Start with every word as a tuple of characters + an end-of-word marker `</w>`.
2. Count every adjacent symbol pair, weighted by word frequency.
3. Merge the most frequent pair everywhere.
4. Repeat `num_merges` times.

Each merge adds exactly one token to the vocabulary.

In [ ]:
import time

def train_bpe(text, num_merges=400, verbose_every=100):
    """Module 2.1's BPE, keyed by the original word so we get an encoding table free."""
    freq = Counter(text.split())                                  # word -> count
    tok = {w: tuple(w) + ("</w>",) for w in freq}                 # word -> current symbols
    merges = []

    for step in range(num_merges):
        # 1. count adjacent pairs, weighted by how often the word appears
        pairs = Counter()
        for w, symbols in tok.items():
            f = freq[w]
            for a, b in zip(symbols, symbols[1:]):
                pairs[(a, b)] += f
        if not pairs:
            break

        # 2. merge the most frequent pair everywhere
        best = max(pairs, key=pairs.get)
        merges.append(best)
        a, b = best
        for w, symbols in tok.items():
            if len(symbols) < 2:
                continue
            out, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                    out.append(a + b); i += 2
                else:
                    out.append(symbols[i]); i += 1
            tok[w] = tuple(out)

        if verbose_every and (step + 1) % verbose_every == 0:
            print(f"  merge {step + 1:>4}: {a!r} + {b!r} -> {(a + b)!r}  ({pairs[best]:,}x)")

    return merges, tok


print("Training BPE on the corpus (this takes a few seconds)...")
t0 = time.time()
merges, word_to_tokens = train_bpe(text, num_merges=400)
print(f"\nDone in {time.time() - t0:.1f}s -- learned {len(merges)} merges.")

print("\nFirst 12 merges (the most common patterns in Shakespeare):")
for i, (a, b) in enumerate(merges[:12]):
    print(f"  {i + 1:>2}. {a!r} + {b!r} -> {(a + b)!r}")

print("\nHow some words ended up tokenized:")
for w in ["the", "lord", "Shakespeare", "unfriendliness"]:
    toks = word_to_tokens.get(w) or "(not in corpus)"
    print(f"  {w!r:>18} -> {toks}")

In [ ]:
# Build the vocabulary: every distinct symbol that survives in the final tokenization.
vocab = sorted({sym for symbols in word_to_tokens.values() for sym in symbols})
stoi = {s: i for i, s in enumerate(vocab)}
itos = {i: s for s, i in stoi.items()}

print(f"BPE vocabulary size: {len(vocab)} tokens")
print(f"  (started from {len(set(text))} distinct characters, +{len(merges)} merges)")

# The comparison that matters: how many tokens does the corpus become?
char_tokens = len(text)
bpe_tokens = sum(len(word_to_tokens[w]) for w in text.split())

print(f"\n{'tokenizer':<14}{'vocab':>8}{'tokens for the corpus':>24}{'chars/token':>14}")
print("-" * 62)
print(f"{'character':<14}{len(set(text)):>8}{char_tokens:>24,}{1.0:>14.2f}")
print(f"{'BPE':<14}{len(vocab):>8}{bpe_tokens:>24,}{char_tokens / bpe_tokens:>14.2f}")
print(f"\nBPE packs {char_tokens / bpe_tokens:.2f}x more text into each token.")
print("With a fixed block_size, that is directly more CONTEXT for the same compute.")

### Why the capstone still uses characters (and when you'd switch)

**Module 5.3 trains on one-character-per-token.** That is a deliberate choice, not an
oversight, and now you have the numbers to see the trade:

| | character-level | BPE (400 merges) | BPE (real, 50k merges) |
|---|---|---|---|
| Vocab | 65 | 463 | ~50,000 |
| Token-table params @ `d_model=128` | 8.3 K | 59 K | **6.4 M** |
| Text per token | 1 char | ~2.3 chars | ~4 chars |

That middle row is `vocab_size × d_model` — counted **once**, because the capstone ties
its embedding and output weights into a single matrix (Module 4.4). Untied, double it.

Look at the third column. A realistic 50k vocabulary would need **6.4 M parameters just
for that one matrix** — six times the *entire* ~1.06 M-parameter capstone model. The
tokenizer's table would dwarf the thing it feeds.

**The rule:** vocabulary size should scale with model size. Tiny models want tiny
vocabularies; a 65-token character vocab is genuinely the right call at 1 M parameters.
Above roughly 100 M parameters the balance flips hard, and every real LLM uses BPE.

You now have both streams. The exercise at the end of this notebook walks you through
swapping the capstone over if you want to see it for yourself.

## 5. Encode the corpus into one flat array

The model doesn't see documents or words. It sees **one long, flat stream of integers**,
and it reads fixed-size windows out of it.

Two implementation details that look like trivia but aren't:

- **`uint16`, not `int64`.** With a vocabulary under 65,536 every token fits in 2 bytes.
  PyTorch's default `int64` would use 8. That's a **4× difference** on a file you will
  read billions of times — and at real scale it decides whether the dataset fits in
  page cache.
- **Store it as a flat file, not a Python list.** Real pipelines write one `.bin` and
  `np.memmap` it, so the OS pages in only the slices being read. A 2 TB dataset is then
  perfectly usable on a machine with 64 GB of RAM.

In [ ]:
import numpy as np

def encode(text, word_to_tokens, stoi):
    """Text -> flat list of token ids, using the table BPE training already gave us."""
    ids = []
    for w in text.split():
        for sym in word_to_tokens[w]:
            ids.append(stoi[sym])
    return ids

t0 = time.time()
ids = encode(text, word_to_tokens, stoi)
stream = np.array(ids, dtype=np.uint16)      # <- 2 bytes per token, not 8
print(f"Encoded {len(stream):,} tokens in {time.time() - t0:.1f}s")

print(f"\n{'representation':<28}{'bytes':>14}{'vs raw text':>14}")
print("-" * 56)
raw = len(text.encode('utf-8'))
print(f"{'raw utf-8 text':<28}{raw:>14,}{'1.00x':>14}")
print(f"{'token stream (uint16)':<28}{stream.nbytes:>14,}{stream.nbytes / raw:>13.2f}x")
print(f"{'token stream (int64)':<28}{stream.nbytes * 4:>14,}{stream.nbytes * 4 / raw:>13.2f}x")

print("\nFirst 20 tokens, and what they decode to:")
print(" ", stream[:20].tolist())
print(" ", [itos[i] for i in stream[:20].tolist()])

### Packing, and the document-boundary question

Our corpus is one continuous play, so the stream is naturally continuous. A real corpus
is millions of *separate* documents, and you have to decide what happens at the seams.

You cannot train on one document per row — documents have wildly different lengths, so
you'd waste most of every batch on padding. Instead you **pack**: concatenate documents
end to end and slice fixed-size windows straight through the joins.

That creates a real problem: a window can straddle two unrelated documents, and the
model is asked to predict the start of a cooking blog from the end of a legal contract.
The standard mitigations:

| Approach | What it does | Cost |
|---|---|---|
| **EOS token between documents** | insert a `<|endoftext|>` marker so the model can *learn* "this is a boundary" | ~free, and universal |
| **Document masking** | mask attention so tokens never attend across a boundary | correct, but needs per-batch masks |
| **Drop the straddling window** | throw away windows that cross a join | simple, wastes data |

Everyone inserts EOS. Frontier training runs increasingly do the attention masking too.
Let's insert the boundary token and watch it appear in the stream.

In [ ]:
# Treat each blank-line-separated block of the play as its own "document".
raw_docs = [d.strip() for d in text.split("\n\n") if d.strip()]
print(f"Split the corpus into {len(raw_docs):,} documents.")
print(f"Document lengths: min={min(len(d.split()) for d in raw_docs)}, "
      f"max={max(len(d.split()) for d in raw_docs)}, "
      f"mean={sum(len(d.split()) for d in raw_docs) / len(raw_docs):.0f} words")
print("-> wildly uneven. This is exactly why we pack instead of padding.\n")

EOS = len(vocab)                       # one new id, appended past the BPE vocabulary
itos[EOS] = "<|endoftext|>"
vocab_size = len(vocab) + 1

packed = []
for d in raw_docs:
    packed.extend(stoi[s] for w in d.split() for s in word_to_tokens[w])
    packed.append(EOS)                 # <- the boundary marker
packed = np.array(packed, dtype=np.uint16)

print(f"Packed stream: {len(packed):,} tokens (vocab_size is now {vocab_size})")
print(f"EOS tokens inserted: {(packed == EOS).sum():,}")

boundary = int(np.argmax(packed == EOS))
window = packed[boundary - 6: boundary + 7]
print("\nA window straddling a document boundary:")
print("  ", [itos[i] for i in window.tolist()])
print("\nWithout that marker the model would have no way to know the topic just reset.")

## 6. Split, then `get_batch`

Now the pipeline's last mile: hold out a validation set, then serve random windows.

**Split *contiguously*, never randomly.** Slicing the stream into small chunks and
shuffling them into train/val is the single easiest way to poison your evaluation:
adjacent chunks share context, so the "held-out" set is full of text the model saw.
Cut once, near the end, and keep the two halves apart. (§8 measures exactly how badly
the random split leaks.)

`get_batch` then does something almost too simple to look right: pick `B` random start
positions, take `T+1` tokens from each, and use `[:-1]` as input and `[1:]` as target.
That's the shifted-target trick from **Module 5.1**, applied to the stream.

In [ ]:
import torch

split_at = int(0.9 * len(packed))
train_data = packed[:split_at]
val_data = packed[split_at:]
print(f"train: {len(train_data):,} tokens   val: {len(val_data):,} tokens")

def get_batch(data, batch_size=4, block_size=16, seed=None):
    """B random windows of length block_size, with targets shifted one to the right."""
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    ix = torch.randint(len(data) - block_size - 1, (batch_size,), generator=g)
    # .astype(np.int64): uint16 saves DISK; the embedding layer needs int64 indices.
    x = torch.stack([torch.from_numpy(data[i:i + block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])
    return x, y

xb, yb = get_batch(train_data, batch_size=4, block_size=16, seed=0)
print(f"\nx: {tuple(xb.shape)}   y: {tuple(yb.shape)}   dtype: {xb.dtype}")

print("\nRow 0 -- watch y be x shifted left by exactly one:")
print("  x:", [itos[i] for i in xb[0].tolist()])
print("  y:", [itos[i] for i in yb[0].tolist()])

print("\nThis (B, T) int64 tensor is precisely what your GPT from Module 4.4 eats.")
print("The pipeline is complete: raw text -> filter -> dedup -> BPE -> pack -> batch.")

## 7. Tokens, not epochs

Everything you have learned about training elsewhere assumes **epochs**: you have a
dataset, you sweep it, you sweep it again, you stop when validation loss turns up.

LLM pretraining does not work like that, and the difference matters enough that it
changes the vocabulary people use.

| | classic ML | LLM pretraining |
|---|---|---|
| Unit of progress | epochs | **tokens seen** |
| Typical passes over data | 10–200 | **~1** |
| Main risk | overfitting | *under*-training (running out of compute) |
| When to stop | val loss turns up | the token budget runs out |

At web scale you have far more text than compute, so you see most documents **once** —
and a model that only sees each example once can barely overfit it. That's why
frontier training curves show train and validation loss falling together, almost on
top of each other, with none of the classic divergence.

This reframing is what **Module 5.7 (scaling laws)** is built on: Chinchilla's answer
to "how big a model should I train?" is stated as a ratio of **parameters to tokens**
(roughly 20 tokens per parameter). Not epochs. Tokens.

> **Watch for the inversion.** Module 5.6 deliberately provokes overfitting on a tiny
> model and a tiny corpus, because it's instructive to *see* the curves separate.
> Don't over-generalize from it: at real scale, with single-pass data, that picture
> mostly doesn't happen. Both facts are true in their own regime.

In [ ]:
# How far does our corpus actually go? The Chinchilla rule of thumb: ~20 tokens/param.
n_tokens = len(packed)
print(f"Our corpus: {n_tokens:,} tokens\n")
print(f"{'model size':>14}{'tokens needed (20x)':>22}{'epochs over our corpus':>26}")
print("-" * 62)
for params, label in [(1e6, "1 M (capstone)"), (125e6, "125 M (GPT-2 small)"),
                      (7e9, "7 B (Llama-7B)"), (70e9, "70 B")]:
    need = 20 * params
    print(f"{label:>14}{need:>22,.0f}{need / n_tokens:>26,.0f}")

print("\nEven the 1M-parameter capstone would want ~20M tokens; we have ~0.4M.")
print("So the capstone necessarily does many passes and CAN overfit -- which is")
print("exactly the regime Module 5.6 explores. Real runs live in the top rows,")
print("where a single pass is all you get.")

## 8. Contamination — proving your held-out set is really held out

Here is the failure that quietly invalidates more reported results than any other.

If text from your test set also appears in your training set, your held-out score
measures **memorization**, not generalization. The model looks brilliant and is merely
reciting. At web scale this happens constantly by accident: benchmark questions get
posted to forums, scraped, and land in the crawl.

The check is the same machinery as dedup, pointed at your own splits: take n-grams
from the validation set and ask how many also occur in training. We use 13-grams,
following the convention in the GPT-3 and Llama contamination analyses.

This is also where §3 finally pays off. The claim there was that *deduplication
protects your evaluation*. Let's prove it, in three measurements:

1. our clean, contiguously-split corpus — the baseline;
2. the same corpus with 30% of documents duplicated, the way a real crawl would have
   them, so a document's twin can land on the other side of the split;
3. the same duplicated corpus, **deduplicated first**, then split.

In [ ]:
def contamination(train_arr, val_arr, n=13, sample=4000, seed=0):
    """Fraction of sampled val n-grams that appear verbatim in train."""
    train_ngrams = {tuple(train_arr[i:i + n].tolist())
                    for i in range(0, len(train_arr) - n, 3)}   # stride 3 to keep it quick
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, len(val_arr) - n, size=min(sample, max(len(val_arr) - n, 1)))
    hits = sum(tuple(val_arr[i:i + n].tolist()) in train_ngrams for i in starts)
    return hits / len(starts)

def pack_docs(docs):
    """Same packing as Section 5, as a function so we can re-pack variants."""
    out = []
    for d in docs:
        out.extend(stoi[s] for w in d.split() for s in word_to_tokens[w])
        out.append(EOS)
    return np.array(out, dtype=np.uint16)

def split_and_measure(docs, label):
    arr = pack_docs(docs)
    cut = int(0.9 * len(arr))
    rate = contamination(arr[:cut], arr[cut:])
    print(f"{label:<38}{rate:>7.1%}   ({len(docs):,} docs, {len(arr):,} tokens)")
    return rate

print(f"{'corpus':<38}{'leaked':>7}")
print("-" * 72)

# 1. Our clean corpus, split contiguously.
split_and_measure(raw_docs, "clean, contiguous split")

# 2. A realistic corpus: 30% of documents appear twice, in no particular order.
rng_py = random.Random(0)
duplicated = raw_docs + rng_py.sample(raw_docs, k=int(0.3 * len(raw_docs)))
rng_py.shuffle(duplicated)
split_and_measure(duplicated, "with 30% duplicates")

# 3. The same corpus, deduplicated BEFORE splitting.
seen_h, deduped = set(), []
for d in duplicated:
    h = hashlib.sha256(d.encode()).hexdigest()
    if h not in seen_h:
        seen_h.add(h)
        deduped.append(d)
split_and_measure(deduped, "same corpus, deduplicated first")

print("""
There it is. Duplicates alone push 13-gram leakage from ~0% to well over 10%: for one
held-out passage in eight, the model has read the exact text during training. Nothing
about the split was careless -- it was still a clean contiguous cut. The corpus was
the problem.

Twenty lines of exact-hash dedup takes it straight back to ~0%.

Run this check whenever you build a split. It is the difference between an evaluation
you can trust and one you cannot -- and you cannot tell the difference by looking at
the loss curve.""")

## Summary

You built the half of LLM training that usually stays invisible:

| Step | What it does | Why it matters |
|---|---|---|
| **Source & mixture** | choose corpora and their proportions | the recipe *is* the model's knowledge |
| **Quality filtering** | cheap heuristics over billions of docs | garbage in, garbage recited |
| **Deduplication** | exact hashing + MinHash near-dup | saves compute, cuts memorization, protects eval |
| **BPE training** | your Module 2.1 tokenizer, on a real corpus | ~2× more text per token |
| **Encoding** | flat `uint16` stream | 4× smaller than `int64`, memmap-able |
| **Packing** | concatenate + EOS boundaries | no padding waste |
| **Split & batch** | contiguous split, shifted windows | `(B, T)` tensors your GPT eats |
| **Contamination check** | n-gram overlap across the split | the difference between a real eval and a lie |

Three things to carry forward:

1. **The capstone's char-level tokenizer is a size-appropriate choice**, and you can
   now defend it with numbers rather than take it on faith.
2. **Split contiguously.** The random-chunk split leaked badly, and it's the one that
   feels more careful.
3. **Think in tokens, not epochs.** Module 5.7 depends on it.

Next, **Module 5.1** asks the question this stream exists to answer: given a window of
tokens, how wrong was the model about the next one?

### 🏋️ Try it yourself

1. **Tune the vocabulary.** Re-run `train_bpe` with `num_merges` of 100, 400, and 1000.
   Plot vocabulary size against the compression ratio (chars per token). The curve
   flattens — find roughly where the extra merges stop paying for themselves.
2. **Swap the capstone over to BPE.** In Module 5.3, replace the character tokenizer
   with the `stream`/`get_batch` built here and set `vocab_size` accordingly. It will
   train — but watch the parameter count jump and the loss numbers shift (cross-entropy
   is per *token*, and your tokens are now bigger, so the numbers aren't comparable to
   the char-level run). Explaining *why* they aren't comparable is the real exercise.
3. **Break the dedup.** Duplicate 20% of the documents in `raw_docs` before packing,
   then re-run the contamination check. How much does the leak rate move?
4. **Filter the real corpus.** Run `quality_report` over `raw_docs` and print which
   ones it drops. Are the rejections fair? Tune a threshold you'd defend, and notice
   how quickly heuristic filtering becomes a judgement call.

In [ ]:
# Task 1 starter: does a bigger vocabulary keep paying off?
import matplotlib.pyplot as plt

sizes, ratios = [], []
for n in [50, 100, 200, 400]:
    m, tk = train_bpe(text, num_merges=n, verbose_every=0)
    v = len({s for syms in tk.values() for s in syms})
    n_tok = sum(len(tk[w]) for w in text.split())
    sizes.append(v); ratios.append(len(text) / n_tok)
    print(f"merges={n:>4}  vocab={v:>4}  chars/token={ratios[-1]:.2f}")

plt.figure(figsize=(6, 3.5))
plt.plot(sizes, ratios, "o-")
plt.xlabel("vocabulary size"); plt.ylabel("chars per token")
plt.title("Diminishing returns on vocabulary size")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()